# 実行準備

## ライブラリのインストール

In [1]:
import sys
import os
import importlib
import re

In [1]:
import json

with open("../data/資格一覧_ver1.00.json", "r", encoding="utf-8-sig") as f:
    certs_json = json.load(f)
    certs = certs_json["qualification"] if isinstance(certs_json, dict) else certs_json

# --- skills.json ---
with open("../data/skillcheck_ver6.00.json", "r", encoding="utf-8-sig") as f:
    skills_json = json.load(f)

In [2]:
skills_json

{'base': [{'No': 1,
   'SubNo': 1,
   'スキルカテゴリ': '行動規範',
   'サブカテゴリ': 'ビジネスマインド',
   'スキルレベル': '★',
   'チェック項目': 'ビジネスにおける「論理とデータの重要性」を認識し、分析的でデータドリブンな考え方に基づき行動できる',
   'BZ': nan,
   'DS': nan,
   'DE': nan,
   '必須スキル': '〇',
   '旧区分': '旧BZ',
   'Unnamed:11': nan},
  {'No': 2,
   'SubNo': 2,
   'スキルカテゴリ': '行動規範',
   'サブカテゴリ': 'ビジネスマインド',
   'スキルレベル': '★',
   'チェック項目': '「目的やゴールの設定がないままデータを分析しても、意味合いが出ない」ことを理解している',
   'BZ': nan,
   'DS': nan,
   'DE': nan,
   '必須スキル': '〇',
   '旧区分': '旧BZ',
   'Unnamed:11': ' '},
  {'No': 3,
   'SubNo': 3,
   'スキルカテゴリ': '行動規範',
   'サブカテゴリ': 'ビジネスマインド',
   'スキルレベル': '★',
   'チェック項目': '課題や仮説を言語化することの重要性を理解している',
   'BZ': nan,
   'DS': nan,
   'DE': nan,
   '必須スキル': '〇',
   '旧区分': '旧BZ',
   'Unnamed:11': nan},
  {'No': 4,
   'SubNo': 4,
   'スキルカテゴリ': '行動規範',
   'サブカテゴリ': 'ビジネスマインド',
   'スキルレベル': '★',
   'チェック項目': '現場に出向いてヒアリングするなど、一次情報に接することの重要性を理解している',
   'BZ': nan,
   'DS': nan,
   'DE': nan,
   '必須スキル': '〇',
   '旧区分': '旧BZ',
   'Unnamed:11': nan},
  {'No

In [3]:

# skills.json は {category: [items]} の構造なので flatten する
skills = []
for category, items in skills_json.items():
    for item in items:
        # 大分類をメタデータとして付与
        item_with_category = {
            **item,
            "category": category
        }
        skills.append(item_with_category)

In [4]:
skills

[{'No': 1,
  'SubNo': 1,
  'スキルカテゴリ': '行動規範',
  'サブカテゴリ': 'ビジネスマインド',
  'スキルレベル': '★',
  'チェック項目': 'ビジネスにおける「論理とデータの重要性」を認識し、分析的でデータドリブンな考え方に基づき行動できる',
  'BZ': nan,
  'DS': nan,
  'DE': nan,
  '必須スキル': '〇',
  '旧区分': '旧BZ',
  'Unnamed:11': nan,
  'category': 'base'},
 {'No': 2,
  'SubNo': 2,
  'スキルカテゴリ': '行動規範',
  'サブカテゴリ': 'ビジネスマインド',
  'スキルレベル': '★',
  'チェック項目': '「目的やゴールの設定がないままデータを分析しても、意味合いが出ない」ことを理解している',
  'BZ': nan,
  'DS': nan,
  'DE': nan,
  '必須スキル': '〇',
  '旧区分': '旧BZ',
  'Unnamed:11': ' ',
  'category': 'base'},
 {'No': 3,
  'SubNo': 3,
  'スキルカテゴリ': '行動規範',
  'サブカテゴリ': 'ビジネスマインド',
  'スキルレベル': '★',
  'チェック項目': '課題や仮説を言語化することの重要性を理解している',
  'BZ': nan,
  'DS': nan,
  'DE': nan,
  '必須スキル': '〇',
  '旧区分': '旧BZ',
  'Unnamed:11': nan,
  'category': 'base'},
 {'No': 4,
  'SubNo': 4,
  'スキルカテゴリ': '行動規範',
  'サブカテゴリ': 'ビジネスマインド',
  'スキルレベル': '★',
  'チェック項目': '現場に出向いてヒアリングするなど、一次情報に接することの重要性を理解している',
  'BZ': nan,
  'DS': nan,
  'DE': nan,
  '必須スキル': '〇',
  '旧区分': '旧BZ',
  'Unnamed:11': na

In [ ]:
from pydantic import BaseModel
import chromadb
from chromadb.config import Settings
from chromadb import PersistentClient
from sentence_transformers import SentenceTransformer
from llama_cpp import Llama
import json
import os

c:\学習\repo_del\cleanrepo\selfstudy_assistance_agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# -----------------------------
# 1. Embedding モデル
# -----------------------------
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# -----------------------------
# 2. Chroma（ローカル永続化）
# -----------------------------
chroma_client = PersistentClient(path="./vectorstore")

collection = chroma_client.get_or_create_collection(
    name="rag_collection",
    metadata={"hnsw:space": "cosine"}
)

# embeddingに強いスキル文書作成（JSON データから Chroma に登録するための文書、メタデータ、ID を作成する関数）
def build_skill_documents(skills_data):
    documents = []
    metadatas = []
    ids = []

    for category, items in skills_data.items():
        for i, skill in enumerate(items):
            name = str(skill.get("No", ""))+"-"+str(skill.get("SubNo", ""))
            level = skill.get("スキルレベル", "")
            level = level.count("★")  # レベルは★の数で表現されていると仮定
            desc = skill.get("チェック項目", "")

            # ★ embedding 用に強いテキストを組み立てる
            doc = f"[カテゴリ: {category}] スキル名: {name}（レベル{level}） 説明: {desc}"

            documents.append(doc)
            metadatas.append({
                "category": category,
                "name": name,
                "level": level
            })
            ids.append(f"{category}_{i}")

    return documents, metadatas, ids

# ① JSON 読み込み
with open("../data/skillcheck_ver6.00.json", "r", encoding="utf-8-sig") as f:
    skills_data = json.load(f)

# ② embedding に強い文書を作る
documents, metadatas, ids = build_skill_documents(skills_data)

# ③ embedding を計算
embeddings = embedder.encode(documents).tolist()

# ④ Chroma に登録
collection.add(
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas,
    ids=ids
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12397.43it/s]


In [7]:
documents

['[カテゴリ: base] スキル名: 1-1（レベル1） 説明: ビジネスにおける「論理とデータの重要性」を認識し、分析的でデータドリブンな考え方に基づき行動できる',
 '[カテゴリ: base] スキル名: 2-2（レベル1） 説明: 「目的やゴールの設定がないままデータを分析しても、意味合いが出ない」ことを理解している',
 '[カテゴリ: base] スキル名: 3-3（レベル1） 説明: 課題や仮説を言語化することの重要性を理解している',
 '[カテゴリ: base] スキル名: 4-4（レベル1） 説明: 現場に出向いてヒアリングするなど、一次情報に接することの重要性を理解している',
 '[カテゴリ: base] スキル名: 5-5（レベル2） 説明: 作業ありきではなく、解決すべき課題（イシュー）ありきで行動できる',
 '[カテゴリ: base] スキル名: 6-6（レベル1） 説明: データを取り扱う人間として相応しい倫理を身に着けている（データのねつ造、改ざん、盗用を行わないなど）',
 '[カテゴリ: base] スキル名: 7-7（レベル1） 説明: データ、AI、機械学習の意図的な悪用（真偽の識別が困難なレベルの画像・音声作成、フェイク情報の作成、Botによる企業・国家への攻撃など）があり得ることを勘案し、技術に関する基礎的な知識と倫理を身につけている',
 '[カテゴリ: base] スキル名: 8-8（レベル1） 説明: データ分析者・利活用者として、データの倫理的な活用上の許容される範囲や、ユーザサイドへの必要な許諾について概ね理解している（直近の個人情報に関する法令：個人情報保護法、EU一般データ保護規則、データポータビリティなど）',
 '[カテゴリ: base] スキル名: 9-1（レベル1） 説明: データや事象の重複に気づくことができる',
 '[カテゴリ: base] スキル名: 10-2（レベル2） 説明: 初見の課題やテーマに対して、検討の抜け漏れや重複をなくすことができる',
 '[カテゴリ: base] スキル名: 11-3（レベル3） 説明: 前例のない課題やテーマであっても、他の事象からの類推などを活用し、検討の抜け漏れや重複をなくすことができる',
 '[カテゴリ: base] スキル名: 12-4（

In [8]:
query = """私は現在、データサイエンス領域で中級レベルのエンジニアとして業務に携わっています。これまでの経験を通じて、データ分析や機械学習モデルの構築、データ基盤の整備など、幅広い領域に関わってきましたが、自分のスキルを体系的に棚卸ししたことはあまりありません。今回、改めて自分がどの程度のスキルを満たしているのかを整理し、今後のキャリア形成に役立てたいと考えています。\n\nまず、データの収集や前処理に関しては、日常的に業務で行っています。ログデータやアプリケーションの利用データを収集し、分析可能な形に整形する作業は一通り経験しています。特に、Python を使ったデータ前処理には慣れており、pandas や numpy を用いたデータ加工、欠損値処理、特徴量生成などは問題なく行えます。また、SQL を使ったデータ抽出も日常的に行っており、複雑な JOIN やサブクエリを含むクエリの作成も対応できます。\n\n統計に関しては、基礎統計量の理解や仮説検定、相関分析、回帰分析など、一般的な分析手法は実務で活用しています。A/B テストの設計や結果の解釈も経験しており、ビジネス上の意思決定に活かしたこともあります。ただし、ベイズ統計や高度な統計モデリングについては、まだ学習途中であり、実務で活用した経験は限定的です。\n\n機械学習に関しては、scikit-learn を用いたモデル構築を中心に経験があります。分類・回帰モデルの構築、ハイパーパラメータチューニング、交差検証、特徴量エンジニアリングなどは一通り実施できます。また、ランダムフォレストや XGBoost などのアンサンブルモデルも扱ったことがあります。深層学習に関しては、TensorFlow や PyTorch を使った簡単なモデルの構築経験はありますが、業務で本格的に活用したことはありません。\n\nデータ基盤やデータエンジニアリングの領域では、ETL パイプラインの構築やデータフローの設計に関わった経験があります。Airflow を使ったバッチ処理のスケジューリングや、クラウド環境（主に Azure）でのデータパイプライン構築を担当したことがあります。Azure Data Factory を使ったデータ連携や、Azure SQL Database を利用したデータマートの構築なども経験しています。ただし、Spark や Hadoop を使った大規模分散処理の経験は限定的で、基礎的な理解に留まっています。\n\nビジネス価値の創出という観点では、分析結果をレポートやダッシュボードとして可視化し、関係者に説明する機会が多くありました。Power BI や Tableau を使ったダッシュボード作成の経験があり、データのストーリーラインを意識した資料作成も得意です。また、プロジェクトの要件定義や関係者とのコミュニケーションにも携わっており、ビジネス側の課題をデータ分析の観点から整理する役割を担うこともあります。\n\nさらに、最近では機械学習モデルの運用（MLOps）にも興味を持ち、モデルのバージョン管理や CI/CD パイプラインの構築について学習を進めています。実務では MLflow を使ったモデル管理を試した程度ですが、今後はより本格的に取り組みたいと考えています。\n\nこのように、データ分析・機械学習・データ基盤・可視化・ビジネス理解など、幅広い領域に触れてきましたが、自分がどの大分類のスキルをどの程度満たしているのかを客観的に把握したいです。データサイエンス協会のスキルチェックリストに基づいて、私が満たしているスキルを整理し、大分類ごとのレベルを可視化してもらえると助かります。"""

prompt = f"""
あなたはデータサイエンス協会のスキルチェックリストに基づいて、
ユーザが満たしているスキルを判定するアセッサーです。

以下のコンテキストには、スキル項目（base/value/DS/DE/fusion）が含まれています。
ユーザの質問内容から、ユーザが「満たしている」と判断できるスキルを抽出し、
必ず次の JSON 形式で返してください。

出力形式（厳守）：

{{
  "base": [
    {{"skill": "スキル名", "level": 数値}},
    ...
  ],
  "value": [...],
  "DS": [...],
  "DE": [...],
  "fusion": [...]
}}

制約：
- JSON 以外の文章は一切出力しない
- もしコンテキストに該当スキルが見つからない場合でも、ユーザの記述内容から明確に判断できる場合はスキルを含めてよい。
- レベルは該当スキルの最大レベルを返す
- 曖昧な場合は含めない
- 絵文字は禁止
- 日本語で返す
- 不要な記号（\\n や *** など）は使わない


# コンテキスト
{context}

# ユーザ質問
{query}

# JSON 出力
"""

NameError: name 'context' is not defined

In [ ]:
contexts

['[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: qualificati

In [ ]:
embedding = embedder.encode(query).tolist()

results = collection.query(
    query_embeddings=[embedding],
    n_results=50,
    include=["documents", "distances"]
    )

docs = []
distances = results.get("distances", [[]])[0] or []
contexts = results.get("documents", [[]])[0] or []

# ③-2 検索結果をスコア値でフィルタリング
min_score = 100
min_doc = None
for doc, score in zip(contexts, distances):
    if score < min_score:
        min_score = score
        min_doc = doc
    if score < 0.4:  # cosine距離なので小さいほど近い
        docs.append(doc)
print(f"最小スコア: {min_score}, フィルタ後ドキュメント数: {len(docs)}")
print(f"最小スコアのドキュメント: {min_doc}")

# fallback
if not docs:
    docs = contexts

# ③ 検索結果をコンテキストとしてまとめる
context = "\n\n".join(docs)

# コンテキストが空の場合のフォールバック
# if len(context == 0:
#     context = "（スキルデータが見つかりませんでした）"


最小スコア: 0.8699633479118347, フィルタ後ドキュメント数: 0
最小スコアのドキュメント: [カテゴリ: value] スキル名: （レベル）。説明: 


In [10]:
documents = results.get("documents", [[]])[0] or []
documents

['[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: value] スキル名: （レベル）。説明: ',
 '[カテゴリ: qualificati

In [33]:
skills_data

{'base': [{'No': 1,
   'SubNo': 1,
   'スキルカテゴリ': '行動規範',
   'サブカテゴリ': 'ビジネスマインド',
   'スキルレベル': '★',
   'チェック項目': 'ビジネスにおける「論理とデータの重要性」を認識し、分析的でデータドリブンな考え方に基づき行動できる',
   'BZ': nan,
   'DS': nan,
   'DE': nan,
   '必須スキル': '〇',
   '旧区分': '旧BZ',
   'Unnamed:11': nan},
  {'No': 2,
   'SubNo': 2,
   'スキルカテゴリ': '行動規範',
   'サブカテゴリ': 'ビジネスマインド',
   'スキルレベル': '★',
   'チェック項目': '「目的やゴールの設定がないままデータを分析しても、意味合いが出ない」ことを理解している',
   'BZ': nan,
   'DS': nan,
   'DE': nan,
   '必須スキル': '〇',
   '旧区分': '旧BZ',
   'Unnamed:11': ' '},
  {'No': 3,
   'SubNo': 3,
   'スキルカテゴリ': '行動規範',
   'サブカテゴリ': 'ビジネスマインド',
   'スキルレベル': '★',
   'チェック項目': '課題や仮説を言語化することの重要性を理解している',
   'BZ': nan,
   'DS': nan,
   'DE': nan,
   '必須スキル': '〇',
   '旧区分': '旧BZ',
   'Unnamed:11': nan},
  {'No': 4,
   'SubNo': 4,
   'スキルカテゴリ': '行動規範',
   'サブカテゴリ': 'ビジネスマインド',
   'スキルレベル': '★',
   'チェック項目': '現場に出向いてヒアリングするなど、一次情報に接することの重要性を理解している',
   'BZ': nan,
   'DS': nan,
   'DE': nan,
   '必須スキル': '〇',
   '旧区分': '旧BZ',
   'Unnamed:11': nan},
  {'No

In [34]:
items

[{'No': 1,
  'SubNo': 1,
  '分類': 'AI実装・運用',
  'スキルカテゴリ': 'AIエージェント',
  'サブカテゴリ': '導入設計',
  'スキルレベル': '★★★',
  'チェック項目': '業務タスクを分析し、エージェントを活用すべき領域と、通常のLLM利用や既存ワークフローで十分な領域を区別して判断できる',
  '必須': '◯'},
 {'No': 2,
  'SubNo': 2,
  '分類': 'AI実装・運用',
  'スキルカテゴリ': 'AIエージェント',
  'サブカテゴリ': '導入設計',
  'スキルレベル': '★★★',
  'チェック項目': '複雑な業務や課題をエージェントが処理可能な単位に分解し、役割やワークフローを設計できる（タスク分割やゴール設定、ワークフロー定義など）',
  '必須': '◯'},
 {'No': 3,
  'SubNo': 3,
  '分類': 'AI実装・運用',
  'スキルカテゴリ': 'AIエージェント',
  'サブカテゴリ': 'アーキテクチャ設計',
  'スキルレベル': '★★',
  'チェック項目': 'AIエージェントの主要な型（Plan-and-Execute、Plan-and-Act、ReAct、Reflexなど）の特徴を理解し、タスク特性に応じて適切な型を選択できる',
  '必須': '◯'},
 {'No': 4,
  'SubNo': 4,
  '分類': 'AI実装・運用',
  'スキルカテゴリ': 'AIエージェント',
  'サブカテゴリ': 'アーキテクチャ設計',
  'スキルレベル': '★★★',
  'チェック項目': 'AIエージェントを実現するための主要な技術要素を体系的に理解し、処理の構成を設計できる（基盤モデル、思考プロセス、ツール利用、メモリ/ナレッジ管理など）',
  '必須': '◯'},
 {'No': 5,
  'SubNo': 5,
  '分類': 'AI実装・運用',
  'スキルカテゴリ': 'AIエージェント',
  'サブカテゴリ': 'アーキテクチャ設計',
  'スキルレベル': '★★★',
  'チェック項目': '業務要件に応じてオーケストレーション型/A2A型な

In [30]:
for category, items in skills_data.items():
    # print(f"カテゴリ: {category}")
    # print(f"スキル数: {len(items)}")
    for skill in items:
        print(f"  - スキル番号: {category+'-'+str(skill.get('No', ''))+'-'+str(skill.get('SubNo', ''))}, レベル: {skill.get('スキルレベル', '')}")

  - スキル番号: base-1-1, レベル: ★
  - スキル番号: base-2-2, レベル: ★
  - スキル番号: base-3-3, レベル: ★
  - スキル番号: base-4-4, レベル: ★
  - スキル番号: base-5-5, レベル: ★★
  - スキル番号: base-6-6, レベル: ★
  - スキル番号: base-7-7, レベル: ★
  - スキル番号: base-8-8, レベル: ★
  - スキル番号: base-9-1, レベル: ★
  - スキル番号: base-10-2, レベル: ★★
  - スキル番号: base-11-3, レベル: ★★★
  - スキル番号: base-12-4, レベル: ★
  - スキル番号: base-13-5, レベル: ★★
  - スキル番号: base-14-6, レベル: ★
  - スキル番号: base-15-7, レベル: ★★
  - スキル番号: base-16-1, レベル: ★
  - スキル番号: base-17-1, レベル: ★
  - スキル番号: base-18-2, レベル: ★
  - スキル番号: base-19-3, レベル: ★★
  - スキル番号: base-20-1, レベル: ★
  - スキル番号: base-21-2, レベル: ★
  - スキル番号: base-22-3, レベル: ★★
  - スキル番号: base-23-4, レベル: ★★★
  - スキル番号: base-24-5, レベル: ★
  - スキル番号: base-25-6, レベル: ★★
  - スキル番号: base-26-7, レベル: ★★★
  - スキル番号: base-27-8, レベル: ★
  - スキル番号: base-28-9, レベル: ★★
  - スキル番号: base-29-10, レベル: ★★
  - スキル番号: base-30-1, レベル: ★
  - スキル番号: base-31-1, レベル: ★
  - スキル番号: base-32-1, レベル: ★
  - スキル番号: base-33-1, レベル: ★
  - スキル番号: value-1-1, レベル: 
  - スキル番